In [ ]:
!git clone --depth 1 https://github.com/trextrader/hotdogornot /kaggle/working/hotdogornot

In [ ]:
%cd /kaggle/working/hotdogornot

In [ ]:
%cd /kaggle/working/hotdogornot/training

In [ ]:
# @title
!pip install -e ".[dev]"

In [ ]:
print("shallow clone already at trextrader HEAD; no pull needed")

In [ ]:
%cd /kaggle/working/hotdogornot
!ls training/data/labeled/embedder
!python -c "import sys; sys.path.insert(0,'training'); from rfconnectorai.data.classes import class_names; print('classes:', class_names('training/configs/classes.yaml'))"

In [ ]:
%%bash
  set -euxo pipefail
  cd /kaggle/working/hotdogornot
  export PYTHONPATH=/kaggle/working/hotdogornot/training
  export PYTHONUNBUFFERED=1

  python -u -m rfconnectorai.data.audit \
      --data-dir training --out docs/DATASET_AUDIT.md

  echo "----- audit summary -----"
  head -40 docs/DATASET_AUDIT.md

  python -u -m rfconnectorai.data.crop_instances \
      --input training/data/labeled/embedder \
      --manifest datasets/rfconnectors/instances.jsonl \
      --out datasets/rfconnectors/crops \
      --mode whole-image --base-dir training

  wc -l datasets/rfconnectors/instances.jsonl

In [ ]:
%%bash
  set -euxo pipefail
  cd /kaggle/working/hotdogornot
  export PYTHONPATH=/kaggle/working/hotdogornot/training
  export PYTHONUNBUFFERED=1

  python -u -m rfconnectorai.data.build_yolo_dataset \
      --input datasets/rfconnectors/instances.jsonl \
      --out datasets/rfconnectors --base-dir training \
      --single-class \
      --taxonomy training/rfconnectorai/specs/connectors.yaml

  cat datasets/rfconnectors/data.yaml   # expect nc: 1, names: [connector]

In [ ]:
# Stage 1: single-class connector localizer (data.yaml nc=1).
# subprocess.Popen + line read = TRUE live output on Kaggle
# (os.system/%%bash don't display; ! breaks on paste-wrap).
import subprocess
cmd = (
    "cd /kaggle/working/hotdogornot && "
    "PYTHONPATH=/kaggle/working/hotdogornot/training PYTHONUNBUFFERED=1 "
    "python -u -m rfconnectorai.detector.train_yolo "
    "--data datasets/rfconnectors/data.yaml --model yolo11n.pt "
    "--epochs 10 --imgsz 640 --batch 96 --device 0 "
    "--dataset-lock datasets/rfconnectors/dataset.lock.json "
    "--out reports/experiments/detector_run_full --artifact-out models/detector"
)
p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end="", flush=True)
p.wait()
print("
[exit code]", p.returncode)


In [ ]:
# CURATED-ONLY 10-class run (635 imgs incl SMA-M).
# subprocess.Popen + line read = TRUE live output on Kaggle
# (os.system/%%bash don't display; ! breaks on paste-wrap).
import subprocess
cmd = (
    "cd /kaggle/working/hotdogornot && "
    "PYTHONPATH=/kaggle/working/hotdogornot/training PYTHONUNBUFFERED=1 "
    "python -u -m rfconnectorai.classifier.train "
    "--data-dir data/labeled/embedder --out-dir models/connector_classifier "
    "--architecture efficientnet_v2_s --input-size 384 "
    "--epochs 50 --batch-size 16 --lr 1e-4"
)
p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end="", flush=True)
p.wait()
print("
[exit code]", p.returncode)


In [ ]:
# Phase 2 staged fine-tune is intentionally SKIPPED under the curated-only
# strategy (2026-05-16). The legacy corpus is synthetic and missing 3 of
# the 9 classes, so there is no separate Phase-1 corpus to warm-start
# from. Cell 9 above is the single curated training run; its output
# models/connector_classifier is consumed by the eval + export cells.
print("Phase 2 skipped: curated-only training (see the cell above).")

In [ ]:
import os, sys, json, torch
from pathlib import Path
os.chdir('/kaggle/working/hotdogornot')
sys.path.insert(0, 'training')
from rfconnectorai.classifier.train import build_model
from rfconnectorai.classifier.dataset import ConnectorFolderDataset, make_eval_transforms
from rfconnectorai.data.classes import class_names
from rfconnectorai.eval.nine_class_report import build_report, render_markdown

cn = class_names('training/configs/classes.yaml')
md = Path('models/connector_classifier')
lab = json.loads((md / 'labels.json').read_text())
arch = lab.get('architecture', 'efficientnet_v2_s')
size = lab.get('input_size', 384)
m = build_model(len(cn), arch)
m.load_state_dict(torch.load(md / 'weights.pt', map_location='cpu'))
m.eval()
ds = ConnectorFolderDataset('data/labeled/embedder', cn, transform=make_eval_transforms(size))
yt, yp = [], []
with torch.no_grad():
    for i in range(len(ds)):
        x, y = ds[i]
        yt.append(y)
        yp.append(int(m(x.unsqueeze(0)).argmax(1)))
        if (i + 1) % 50 == 0:
            print(f'scored {i+1}/{len(ds)}', flush=True)
rep = build_report(yt, yp, cn)
out = Path('reports/experiments/classifier_9class'); out.mkdir(parents=True, exist_ok=True)
(out / 'REPORT.md').write_text(render_markdown(rep))
print(render_markdown(rep))
print('NOTE: scored over ALL imgs incl. training data -> headline accuracy '
      'optimistic; read the CONFUSION STRUCTURE, not the number.')


In [ ]:
# Export ONNX, drop bump_version's byte-identical copies, zip (RELATIVE
# paths so extraction never nests content/hotdogornot or spawns dups),
# then download. ~160MB (1 .pt + 1 .onnx) instead of ~560MB.
!python -m rfconnectorai.classifier.export_onnx     --model-dir /kaggle/working/hotdogornot/models/connector_classifier     --output /kaggle/working/hotdogornot/models/connector_classifier/classifier.onnx

!cd /kaggle/working/hotdogornot/models/connector_classifier && rm -f     weights.0001.pt weights.latest.pt     weights.0001.onnx weights.latest.onnx weights.onnx

!mkdir -p /kaggle/working/hotdogornot/reports/experiments/classifier_9class
!cp /kaggle/working/hotdogornot/models/connector_classifier/metrics.json     /kaggle/working/hotdogornot/models/connector_classifier/version.json     /kaggle/working/hotdogornot/models/connector_classifier/labels.json     /kaggle/working/hotdogornot/reports/experiments/classifier_9class/

!rm -f /kaggle/working/classifier_9class.zip
!cd /kaggle/working/hotdogornot && zip -r /kaggle/working/classifier_9class.zip     models/connector_classifier reports/experiments/classifier_9class

print("Artifact is under /kaggle/working/ -> download it from the "
      "notebook Output panel, or use Save Version to snapshot it.")


In [ ]:
# Detector run: zip with RELATIVE paths (no /kaggle/working/hotdogornot nesting)
# and exclude weights/last.pt (byte-identical to best.pt).
!rm -f /kaggle/working/detector_run_full.zip
!cd /kaggle/working/hotdogornot && zip -r /kaggle/working/detector_run_full.zip     reports/experiments/detector_run_full models/detector/best.pt     -x '*/weights/last.pt'

print("Artifact is under /kaggle/working/ -> download it from the "
      "notebook Output panel, or use Save Version to snapshot it.")


In [ ]:
# Check which runs exist and their sizes
!ls -la /kaggle/working/hotdogornot/reports/experiments/
!wc -l /kaggle/working/hotdogornot/reports/experiments/detector_run_full/ultralytics/connector/results.csv
!cat /kaggle/working/hotdogornot/reports/experiments/detector_run_full/ultralytics/connector/results.csv
